# Phase 0B: Data Audit Report
**Dataset:** NIH ChestX-ray14 & VinDr-CXR (Baseline Preparation)

## 1. Data Leakage Risk Report
**Patient-level Leakage:** 
Chest X-ray datasets frequently contain multiple images (studies) per patient taken over time. A naive random split across all images will result in data leakage, where images from the same patient appear in both the training and test sets. This falsely inflates performance metrics since the model learns patient-specific anatomical features or medical devices rather than generalized disease features.

**Mitigation Plan:**
* Splitting must be strictly handled at the **patient level** using the `Patient ID` metadata.
* Use the official standardized splits provided by the NIH dataset creators to ensure benchmarking validity and completely prevent overlap: `train_val_list.txt` for training and validation, and `test_list.txt` exclusively for the held-out test set. 

## 2. Label Quality Note
**Silver-Standard Labels (NLP-Mined):**
The labels in the NIH ChestX-ray14 dataset were generated by applying an NLP algorithm to the corresponding radiological text reports. They were not manually drawn or validated by radiologists.

**Why this matters:**
This creates a "silver-standard" benchmark. NLP tools inherently produce false positives (e.g., misinterpreting negation or historical references like "no pneumonia") and false negatives. 
* **Noisy/Ambiguous Labels:** Pathologies that are visually subtle or semantically overlapping in text descriptions (such as "Infiltration" vs. "Consolidation" or "Pneumonia") tend to exhibit the highest noise. "Mass" vs "Nodule" boundaries are also heavily subject to interpretation.

## 3. Class Imbalance Plan
Medical datasets inherently suffer from long-tailed class distributions where normal and prevalent states (like Cardiomegaly) dominate, while severe or rare findings (like Hernia) are severely underrepresented.

**Prevalence and Weighting:**
Per-label prevalence must be computed during dataloader initialization using the training set only. 
To counteract the massive imbalance during training, we apply positive class weighting in the loss function `BCEWithLogitsLoss(pos_weight=...)`.

**Formula:** For a given class $c$:
`pos_weight_c = negative_count_c / positive_count_c`

## 4. Subgroup Audit Plan
Performance must be monitored across demographic and technical metadata to ensure fairness and robustness:
* **Age & Sex:** Compute disparate impact and stratification metrics based on patient Age and Sex to prevent undetected bias.
* **View Position (PA vs. AP):** Post-Anterior (PA) and Antero-Posterior (AP) views magnify the heart differently. Models may spuriously correlate AP views (often taken in portable settings for sicker patients) with severity.
* **Lateral Issues:** Ensure lateral scans are uniformly discarded or robustly handled per the Out-of-Distribution (OOD) pipeline if accidentally included.

## 5. Phase 1 Warnings for Codex
**Attention Codex — Critical Implementation Rules:**
* **Do NOT use Softmax:** Pathologies are not mutually exclusive. A patient can have multiple findings. Ensure standard multi-label handling.
* **Loss Function strictly BCEWithLogitsLoss:** Ensure the classification head outputs raw logits. Pass logits directly to `BCEWithLogitsLoss` which is numerically stable. **Do not apply a Sigmoid before the loss function.**
* **Sigmoid is for inference only:** Only apply `torch.sigmoid()` when converting network outputs to probabilities for evaluation or UI visualization. 
* **No test-set tuning:** Do not perform any threshold or temperature tuning on the test set. All calibrations and hyperparameter choices happen exclusively on the validation set.
* **NIH labels are global only:** Do not treat NIH image-level labels as ground truth for localization (bounding boxes). Phase 3 addresses VinDr-CXR for spatial localization metrics.